# LLM Arduino Benchmark — Stage 0 + Stage 1
**Ground Truth Linking + Dictionary Audit and Lookup Class**

Run cells top to bottom. Each section is self-contained.

---

## 0. Installs and Imports

In [ ]:
import sys
!{sys.executable} -m pip install openpyxl

In [ ]:
import pandas as pd
import json
import os
import re
import random
import csv
from tqdm import tqdm
from openai import OpenAI

print('All imports OK')

## 1. Configuration — Set Your Paths Here

In [ ]:
from pathlib import Path
import os


# ── PROJECT ROOT SETUP ──────────────────────────────────────────
PROJECT_ROOT = Path("/Users/anonymous_user/Desktop/text-to-arduino")  
OUTPUT_ROOT = PROJECT_ROOT / "outputs"

DATA_DIR = PROJECT_ROOT / "data"
OUTPUT_DIR =  OUTPUT_ROOT / "benchmark_outputs"

DICT_PATH = DATA_DIR / "components_dictionary.xlsx"
PROJ_PATH = DATA_DIR / "projects_metadata.xlsx"
CODE_BASE_DIR = DATA_DIR / "organized_data" / "codes"

# ── VERIFY PATHS ────────────────────────────────────────────────
print("Dictionary exists:", DICT_PATH.exists())
print("Projects exists:", PROJ_PATH.exists())
print("Code dir exists:", CODE_BASE_DIR.exists())

os.makedirs(OUTPUT_DIR, exist_ok=True)

OPENAI_API_KEY = 'REDACTED_API_KEY'                        # your OpenAI key for linking script

# ────────────────────────────────────────────────────────────────

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
print(f'Output directory: {OUTPUT_DIR}')

---
## STAGE 1 — Dictionary Audit and Lookup Class
> Do this BEFORE Stage 0 so the HardwareDictionary is ready for the linking script

### 1.1 Load and Preview the Dictionary

In [ ]:
import os
os.getcwd()

In [ ]:
os.chdir('/Users/anonymous_user/Desktop/text-to-arduino')

In [ ]:
dict_df = pd.read_excel(DICT_PATH)
print(f'Total components: {len(dict_df)}')
print(f'Columns: {list(dict_df.columns)}')
dict_df.head(3)

### 1.2 Audit All 169 Components for Issues

In [ ]:
import json
import pandas as pd

issues = []

for _, row in dict_df.iterrows():
    cid = row['Component_Id']
    name = row['Component name']
    comp_type = str(row.get('Component Type', '')).strip()
    category = str(row.get('Category', '')).strip()

    # -----------------------------
    # Component classification rules
    # -----------------------------
    is_board = comp_type == 'Microcontroller' or category == 'Development Board'
    requires_library = comp_type in ['Sensor', 'Actuator', 'Communication']
    requires_functions = comp_type in ['Sensor', 'Actuator', 'Communication']

    # -----------------------------
    # Check 1: Pin Description must be valid JSON
    # -----------------------------
    try:
        pins = json.loads(row['Pin Description'])
    except Exception as e:
        issues.append({
            'id': cid,
            'name': name,
            'issue': f'Invalid Pin Description JSON: {e}'
        })
        continue

    # -----------------------------
    # Check 2: I2C/SPI components must have libraries
    # (ONLY for components that require libraries)
    # -----------------------------
    bus_roles = [p.get('bus_role', '') for p in pins if p.get('bus_role')]
    has_i2c = any('I2C' in str(r) for r in bus_roles)
    has_spi = any('SPI' in str(r) for r in bus_roles)

    lib_missing = (
        pd.isna(row['Associated Libraries']) or
        str(row['Associated Libraries']).strip() in ['None', '', 'nan', 'N/A']
    )

    if requires_library and not is_board:
        if (has_i2c or has_spi) and lib_missing:
            issues.append({
                'id': cid,
                'name': name,
                'issue': 'I2C/SPI interface but no Associated Libraries listed'
            })

    # -----------------------------
    # Check 3: Function Names must exist
    # (ONLY for sensors/actuators/communication modules)
    # -----------------------------
    if requires_functions:
        if pd.isna(row['Function Names']) or str(row['Function Names']).strip() in ['', 'N/A', 'nan']:
            issues.append({
                'id': cid,
                'name': name,
                'issue': 'Missing Function Names'
            })

    # -----------------------------
    # Check 4: Notes should exist
    # -----------------------------
    if pd.isna(row['Notes']) or str(row['Notes']).strip() in ['', 'nan']:
        issues.append({
            'id': cid,
            'name': name,
            'issue': 'Missing Notes field'
        })

    # -----------------------------
    # Check 5: Component name should not be empty
    # -----------------------------
    if pd.isna(row['Component name']) or str(row['Component name']).strip() == '':
        issues.append({
            'id': cid,
            'name': name,
            'issue': 'Missing Component name'
        })


print('\n=== AUDIT COMPLETE ===')
print(f'Components checked: {len(dict_df)}')
print(f'Issues found: {len(issues)}')

if issues:
    print('\n--- Issues ---')
    for i in issues:
        print(f"  ID {i['id']} ({i['name']}): {i['issue']}")
    issues_df = pd.DataFrame(issues)
    issues_df.to_csv(f'{OUTPUT_DIR}/dictionary_audit_issues.csv', index=False)
    print(f'\nSaved to {OUTPUT_DIR}/dictionary_audit_issues.csv')
else:
    print('No issues found — dictionary is clean!')

### 1.3 Fix Issues
> Open `dictionary_audit_issues.csv`, fix each row in your Excel file, then reload and re-run the audit above until zero issues.

### 1.4 Build the HardwareDictionary Class

In [ ]:
class HardwareDictionary:
    def __init__(self, xlsx_path):
        df = pd.read_excel(xlsx_path)
        self.db = {}
        self.alias_index = {}  # name/version string -> component_id

        for _, row in df.iterrows():
            cid = int(row['Component_Id'])

            # Parse libraries
            lib_raw = str(row['Associated Libraries'])
            libs = [] if lib_raw.strip() in ['None', 'nan', ''] else [l.strip() for l in lib_raw.split(',')]

            # Parse function names
            fn_raw = str(row['Function Names'])
            fns = [] if fn_raw.strip() in ['None', 'nan', ''] else [f.strip() for f in fn_raw.split(',')]

            # Parse versions
            ver_raw = str(row['Common versions'])
            versions = [] if ver_raw.strip() in ['None', 'nan', ''] else [v.strip() for v in ver_raw.split(',')]

            self.db[cid] = {
                'id': cid,
                'name': str(row['Component name']).strip(),
                'type': str(row['Component Type']).strip(),
                'category': str(row['Category']).strip(),
                'subcategory': str(row['Sub-category']).strip(),
                'versions': versions,
                'libraries': libs,
                'pin_type': str(row['Pin Type']).strip(),
                'pins': json.loads(row['Pin Description']),
                'use_cases': str(row['Typical Use Cases']).strip(),
                'functions': fns,
                'notes': str(row['Notes']).strip()
            }

            # Build alias index: component name + all version strings
            for variant in [row['Component name']] + versions:
                key = str(variant).strip().lower()
                if key:
                    self.alias_index[key] = cid

    def lookup_by_id(self, cid):
        return self.db.get(int(cid))

    def lookup_by_name(self, name):
        cid = self.alias_index.get(str(name).strip().lower())
        return self.db[cid] if cid else None

    def get_prompt_context(self, cid):
        entry = self.lookup_by_id(cid)
        if not entry:
            return f'[WARNING: Component ID {cid} not found in dictionary]'
        return json.dumps({
            'name': entry['name'],
            'libraries': entry['libraries'],
            'pin_type': entry['pin_type'],
            'pins': entry['pins'],
            'functions': entry['functions'],
            'notes': entry['notes']
        }, indent=2)

    def get_canonical_names_list(self):
        """Returns list of {id, name} dicts for use in LLM prompts"""
        return [{'id': e['id'], 'name': e['name']} for e in self.db.values()]

    def get_library_to_ids_map(self):
        """Returns {library_name: [component_ids]} — used for ground truth linking"""
        lib_map = {}
        for cid, entry in self.db.items():
            for lib in entry['libraries']:
                lib_map.setdefault(lib, []).append(cid)
        return lib_map

print('HardwareDictionary class defined.')

### 1.5 Test the Dictionary

In [ ]:
# Load the dictionary
dictionary = HardwareDictionary(DICT_PATH)

# Test 1: Lookup all components by ID — zero failures expected
failures = []
for cid in dictionary.db.keys():
    if dictionary.lookup_by_id(cid) is None:
        failures.append(cid)
print(f'ID lookup test: {len(dictionary.db)} components, {len(failures)} failures')

# Test 2: Lookup by name
sample_names = list(dictionary.alias_index.keys())[:10]
print(f'\nName lookup test (first 10 aliases):')
for name in sample_names:
    result = dictionary.lookup_by_name(name)
    status = f"-> ID {result['id']}: {result['name']}" if result else '-> NOT FOUND'
    print(f'  "{name}" {status}')

# Test 3: Unknown component returns None cleanly
unknown = dictionary.lookup_by_name('xyz unknown sensor 9999')
print(f'\nUnknown component lookup: {unknown}  (should be None)')

# Test 4: Prompt context for first 3 components
print(f'\nPrompt context sample (Component_Id 1):')
print(dictionary.get_prompt_context(1))

In [ ]:
# Test 5: Library-to-component mapping
lib_map = dictionary.get_library_to_ids_map()
print(f'Total unique libraries in dictionary: {len(lib_map)}')
print('\nLibrary -> Component IDs (first 10):')
for lib, ids in list(lib_map.items())[:10]:
    names = [dictionary.lookup_by_id(i)['name'] for i in ids]
    print(f'  {lib}: {names}')

---
## STAGE 0 — Build Ground Truth Linking
> Map each of the 435 projects to canonical component IDs from the dictionary

### 0.1 Load and Preview Project Metadata

In [ ]:
proj_df = pd.read_excel(PROJ_PATH)
print(f'Total projects: {len(proj_df)}')
print(f'Columns: {list(proj_df.columns)}')
proj_df.head(3)

### 0.2 Step 1: Library-Based Linking (Fast, No API Cost)
> Use Libraries_Used column as the most reliable signal first

In [ ]:
lib_map = dictionary.get_library_to_ids_map()

def link_by_libraries(libraries_used_str):
    """
    Given a comma-separated string of libraries from project_metadata,
    return all component IDs that use those libraries.
    """
    if pd.isna(libraries_used_str) or str(libraries_used_str).strip() in ['None', 'nan', '']:
        return [], {}

    libs = [l.strip() for l in str(libraries_used_str).split(',')]
    matched_ids = []
    confidence = {}

    for lib in libs:
        if lib in lib_map:
            for cid in lib_map[lib]:
                if cid not in matched_ids:
                    matched_ids.append(cid)
                    confidence[cid] = 'high'  # library match is high confidence

    return matched_ids, confidence

# Test on first 5 projects
print('Library-based linking test (first 5 projects):')
for _, row in proj_df.head(5).iterrows():
    ids, conf = link_by_libraries(row['Libraries_Used'])
    names = [dictionary.lookup_by_id(i)['name'] for i in ids]
    print(f"  Project {row['Project_ID']}: Libraries='{row['Libraries_Used']}' -> IDs={ids} {names}")

### 0.3 Step 2: LLM-Based Linking for Remaining Components
> For components that don't need libraries (use native functions like digitalRead, pulseIn), use GPT-4o to identify them from the description text

In [ ]:
LINKING_SYSTEM = """You are a hardware component identification expert for Arduino projects.
Given a project description and canonical component list, identify which components are used.
Output ONLY valid JSON — no explanation, no markdown backticks:
{
  "canonical_ids": [1, 15, 42],
  "confidence": {"1": "high", "15": "medium", "42": "low"},
  "reasoning": "brief explanation of each match"
}"""

def llm_link_components(project_row, canonical_list, already_linked_ids, client):
    """
    Use GPT-4o to identify additional components beyond what library matching found.
    already_linked_ids: IDs already found via library matching (skip these)
    """
    # Filter canonical list to exclude already-linked ones
    remaining = [c for c in canonical_list if c['id'] not in already_linked_ids]

    user_prompt = f"""Canonical component list (only select from these):
{json.dumps(remaining, indent=2)}

Project details:
- Description: {project_row.get('Description', '')}
- Components visible in image: {project_row.get('Components Visible', '')}
- Components described in code: {project_row.get('Components in Code', '')}
- Libraries used: {project_row.get('Libraries_Used', '')}
- Board: {project_row.get('Board', '')}

Note: Components already identified via library matching (do NOT include): {already_linked_ids}
Only identify hardware components (sensors, actuators, modules). 
Do NOT include Arduino boards, jumper wires, breadboards, resistors, or batteries."""

    try:
        resp = client.chat.completions.create(
            model='gpt-4o',
            temperature=0,
            messages=[
                {'role': 'system', 'content': LINKING_SYSTEM},
                {'role': 'user', 'content': user_prompt}
            ]
        )
        raw = resp.choices[0].message.content.strip()
        parsed = json.loads(raw)
        return parsed.get('canonical_ids', []), parsed.get('confidence', {}), parsed.get('reasoning', '')
    except Exception as e:
        return [], {}, f'ERROR: {e}'

print('LLM linking function defined.')

### 0.4 Run Full Linking on All 435 Projects

In [ ]:
# FQBN mapping for compilation later
FQBN_MAP = {
    'Arduino Uno': 'arduino:avr:uno',
    'Arduino Uno R4 Wifi Module': 'arduino:renesas_uno:unor4wifi',
    'Arduino Uno R4 WiFi': 'arduino:renesas_uno:unor4wifi',
    'Arduino Mega': 'arduino:avr:mega',
    'Arduino Mega 2560': 'arduino:avr:mega',
    'Arduino Nano': 'arduino:avr:nano',
}

client = OpenAI()  
canonical_list = dictionary.get_canonical_names_list()
ground_truth = []

print(f'Starting linking for {len(proj_df)} projects...')
print('This will make one API call per project. Estimated cost: ~$2-4 for all 435 projects.')
print()

for _, row in tqdm(proj_df.iterrows(), total=len(proj_df)):
    pid = row['Project_ID']

    # Step 1: Library-based linking (free, instant)
    lib_ids, lib_confidence = link_by_libraries(row['Libraries_Used'])

    # Step 2: LLM-based linking for remaining components
    llm_ids, llm_confidence, reasoning = llm_link_components(
        row, canonical_list, lib_ids, client
    )

    # Merge results
    all_ids = list(set(lib_ids + llm_ids))
    all_confidence = {**{str(i): 'high' for i in lib_ids}, **{str(i): llm_confidence.get(i, 'medium') for i in llm_ids}}

    # Parse libraries for ground truth
    libs_raw = str(row.get('Libraries_Used', ''))
    gt_libraries = [] if libs_raw.strip() in ['None', 'nan', ''] else [l.strip() for l in libs_raw.split(',') if l.strip() != 'None']

    # Board FQBN
    board = str(row.get('Board', 'Arduino Uno')).strip()
    fqbn = FQBN_MAP.get(board, 'arduino:avr:uno')

    # Code path
    code_path = str(row.get('Code Path', '')).strip()

    record = {
        'project_id': int(pid),
        'description': str(row.get('Description', '')).strip(),
        'code_path': code_path,
        'board': board,
        'board_fqbn': fqbn,
        'complexity': str(row.get('Complexity', 'Low')).strip(),
        'ground_truth_component_ids': all_ids,
        'ground_truth_libraries': gt_libraries,
        'lib_linked_ids': lib_ids,
        'llm_linked_ids': llm_ids,
        'linking_confidence': all_confidence,
        'llm_reasoning': reasoning,
        'manually_verified': False
    }
    ground_truth.append(record)

# Save raw output
gt_path = f'{OUTPUT_DIR}/ground_truth_raw.json'
with open(gt_path, 'w') as f:
    json.dump(ground_truth, f, indent=2)

print(f'\nDone! Saved {len(ground_truth)} records to {gt_path}')

### 0.5 Inspect the Linking Results

In [ ]:
# Summary statistics
total = len(ground_truth)
has_components = sum(1 for r in ground_truth if len(r['ground_truth_component_ids']) > 0)
empty = total - has_components
avg_components = sum(len(r['ground_truth_component_ids']) for r in ground_truth) / total

print(f'Total projects: {total}')
print(f'Projects with at least 1 component linked: {has_components}')
print(f'Projects with zero components linked: {empty}  <- review these')
print(f'Average components per project: {avg_components:.2f}')

# Complexity distribution
complexity_counts = {}
for r in ground_truth:
    c = r['complexity']
    complexity_counts[c] = complexity_counts.get(c, 0) + 1
print(f'\nComplexity distribution: {complexity_counts}')

# Show first 3 records
print('\n--- First 3 ground truth records ---')
for r in ground_truth[:3]:
    names = [dictionary.lookup_by_id(i)['name'] for i in r['ground_truth_component_ids']]
    print(f"  Project {r['project_id']}: IDs={r['ground_truth_component_ids']} -> {names}")
    print(f"    Libraries: {r['ground_truth_libraries']}")
    print(f"    Complexity: {r['complexity']}, Board: {r['board']}")
    print()

In [ ]:
import json, pandas as pd

# Load the latest ground truth
with open(f'{OUTPUT_DIR}/ground_truth_raw.json') as f:
    ground_truth = json.load(f)

# Find projects with zero components AND not manually verified
zero_component_projects = [
    r for r in ground_truth 
    if len(r['ground_truth_component_ids']) == 0 
    and not r.get('out_of_scope', False)
    and not r.get('manually_verified', False)
]

print(f'Projects with zero components and not manually verified: {len(zero_component_projects)}')

if zero_component_projects:
    print('\nThese need manual review:')
    for r in zero_component_projects[:10]:
        print(f"  Project {r['project_id']}: {r['description'][:80]}...")
    if len(zero_component_projects) > 10:
        print(f'  ... and {len(zero_component_projects) - 10} more')

    # Update zero_component_projects.csv
    zero_df = pd.DataFrame([{
        'project_id':  r['project_id'],
        'description': r['description'][:150],
        'complexity':  r['complexity'],
        'board':       r['board'],
        'libraries':   ', '.join(r['ground_truth_libraries']),
        'reasoning':   r.get('llm_reasoning', '')
    } for r in zero_component_projects])

    zero_df.to_csv(f'{OUTPUT_DIR}/zero_component_projects.csv', index=False)
    print(f'\nUpdated {OUTPUT_DIR}/zero_component_projects.csv')
else:
    print('\nNo unresolved zero-component projects. You are good to go!')
    # Clear the old csv so it is not misleading
    pd.DataFrame().to_csv(f'{OUTPUT_DIR}/zero_component_projects.csv', index=False)
    print('zero_component_projects.csv cleared.')

In [ ]:
# Find projects where board is nan or empty
nan_boards = [
    r for r in ground_truth 
    if not r['board'] or r['board'].strip() in ['nan', '', 'None']
]

print(f'Projects with missing/nan board: {len(nan_boards)}')
for r in nan_boards:
    print(f"  Project {r['project_id']}: '{r['board']}' | {r['description'][:80]}")

In [ ]:
# # Load zero-component project IDs from CSV
# zero_df  = pd.read_csv(f'{OUTPUT_DIR}/zero_component_projects.csv')
# zero_ids = set(zero_df['project_id'].astype(int).tolist())
# print(f'Re-running linking for {len(zero_ids)} projects from zero_component_projects.csv')

# canonical_list = dictionary.get_canonical_names_list()
# retry_results  = {}

# RETRY_SYSTEM = """You are an expert Arduino hardware engineer.
# Your job is to identify ALL physical hardware components in a project.
# Many Arduino components require NO libraries — they use native functions
# like digitalRead(), analogRead(), pulseIn(), tone(), digitalWrite().
# These are still real components and you MUST identify them.

# Examples of no-library components:
# - Push buttons, tactile switches  -> digitalRead()
# - LEDs                            -> digitalWrite(), analogWrite()
# - Potentiometers                  -> analogRead()
# - Passive buzzers                 -> tone()
# - HC-SR04 ultrasonic sensor       -> pulseIn(), digitalWrite()
# - IR sensors                      -> digitalRead()
# - LDR / photoresistors            -> analogRead()
# - Relay modules (basic)           -> digitalWrite()

# Output ONLY valid JSON, no explanation, no markdown:
# {
#   "canonical_ids": [5, 12, 34],
#   "confidence": {"5": "high", "12": "high", "34": "medium"},
#   "reasoning": "LED -> ID 5 because..."
# }
# If you genuinely cannot identify any components, return:
# {"canonical_ids": [], "confidence": {}, "reasoning": "no match"}"""

# for record in tqdm([r for r in ground_truth if r['project_id'] in zero_ids]):
#     pid = record['project_id']
#     user_prompt = f"""Canonical component list:
# {json.dumps(canonical_list, indent=2)}

# Project details:
# - Description: {record['description']}
# - Libraries used: {record['ground_truth_libraries']}
# - Board: {record['board']}

# IMPORTANT: This project uses NO external libraries.
# Identify components that work with native Arduino functions only."""

#     try:
#         resp = client.chat.completions.create(
#             model='gpt-4o', temperature=0,
#             messages=[
#                 {'role': 'system', 'content': RETRY_SYSTEM},
#                 {'role': 'user',   'content': user_prompt}
#             ]
#         )
#         raw    = resp.choices[0].message.content.strip()
#         parsed = json.loads(raw)
#         retry_results[pid] = {
#             'canonical_ids': parsed.get('canonical_ids', []),
#             'confidence':    parsed.get('confidence', {}),
#             'reasoning':     parsed.get('reasoning', '')
#         }
#     except Exception as e:
#         print(f'  Project {pid} failed: {e}')
#         retry_results[pid] = {'canonical_ids': [], 'confidence': {}, 'reasoning': f'ERROR: {e}'}

# # Patch ground_truth in place
# patched    = 0
# still_zero = 0

# for record in ground_truth:
#     pid = record['project_id']
#     if pid not in retry_results:
#         continue
#     new_ids = retry_results[pid]['canonical_ids']
#     if new_ids:
#         record['ground_truth_component_ids'] = new_ids
#         record['llm_linked_ids']             = new_ids
#         record['linking_confidence']         = retry_results[pid]['confidence']
#         record['llm_reasoning']              = retry_results[pid]['reasoning']
#         patched += 1
#     else:
#         still_zero += 1

# # Save updated ground truth
# with open(f'{OUTPUT_DIR}/ground_truth_raw.json', 'w') as f:
#     json.dump(ground_truth, f, indent=2)

# print(f'\nRetry complete:')
# print(f'  Successfully linked:    {patched}')
# print(f'  Still zero after retry: {still_zero}  <- manually link these')
# print(f'ground_truth_raw.json updated.')

### 0.6 Manual Verification — Select 50-Project Pilot Set

In [ ]:
# Stratified sample: pick projects across all complexity levels
random.seed(42)

by_complexity = {'Low': [], 'Medium': [], 'High': []}
for r in ground_truth:
    c = r['complexity']
    if c in by_complexity:
        by_complexity[c].append(r)
    else:
        by_complexity['Low'].append(r)  # default unknown complexity to Low

# Proportional sampling for 50 projects
pilot_set = []
for level, items in by_complexity.items():
    n = max(1, round(50 * len(items) / len(ground_truth)))
    sampled = random.sample(items, min(n, len(items)))
    pilot_set.extend(sampled)
    print(f'  {level}: {len(items)} total -> {len(sampled)} sampled')

# Trim to exactly 50
pilot_set = pilot_set[:50]
print(f'\nTotal pilot set: {len(pilot_set)} projects')

# Save pilot set
pilot_path = f'{OUTPUT_DIR}/pilot_50.json'
with open(pilot_path, 'w') as f:
    json.dump(pilot_set, f, indent=2)
print(f'Saved to {pilot_path}')

In [ ]:
# Generate a manual verification checklist (open in Excel to fill in)
verification_rows = []
for r in pilot_set:
    names = [dictionary.lookup_by_id(i)['name'] for i in r['ground_truth_component_ids']]
    verification_rows.append({
        'project_id': r['project_id'],
        'description': r['description'][:150],
        'complexity': r['complexity'],
        'board': r['board'],
        'ground_truth_libraries': ', '.join(r['ground_truth_libraries']),
        'linked_component_ids': str(r['ground_truth_component_ids']),
        'linked_component_names': ', '.join(names),
        'llm_reasoning': r.get('llm_reasoning', '')[:200],
        'VERIFIED_CORRECT': '',    # <- YOU FILL THIS IN: YES / NO
        'CORRECT_IDS_IF_WRONG': '',  # <- fill in if NO
        'NOTES': ''
    })

ver_df = pd.DataFrame(verification_rows)
ver_path = f'{OUTPUT_DIR}/verification_checklist.xlsx'
ver_df.to_excel(ver_path, index=False)
print(f'Verification checklist saved to {ver_path}')
print(f'Open this file, check each project against its .txt code file, fill in VERIFIED_CORRECT column.')
print(f'Then run the next cell to update ground_truth_raw.json with your corrections.')

In [ ]:
import json, pandas as pd

# Load ground truth and existing checklist
with open(f'{OUTPUT_DIR}/ground_truth_raw.json') as f:
    ground_truth = json.load(f)

existing_checklist = pd.read_excel(f'{OUTPUT_DIR}/verification_checklist.xlsx')

# Get project IDs already in the checklist
already_verified_ids = set(existing_checklist['project_id'].astype(int).tolist())
print(f'Projects already in verification_checklist.xlsx: {len(already_verified_ids)}')

# Find projects in ground_truth_raw that are NOT in the checklist
# Exclude out_of_scope projects
not_in_checklist = [
    r for r in ground_truth
    if r['project_id'] not in already_verified_ids
    and not r.get('out_of_scope', False)
]

print(f'Projects not in checklist: {len(not_in_checklist)}')

if not_in_checklist:
    rows = []
    for r in not_in_checklist:
        names = [dictionary.lookup_by_id(i)['name'] for i in r['ground_truth_component_ids'] if dictionary.lookup_by_id(i)]
        rows.append({
            'project_id':             r['project_id'],
            'description':            r['description'][:150],
            'complexity':             r['complexity'],
            'board':                  r['board'],
            'ground_truth_libraries': ', '.join(r['ground_truth_libraries']),
            'linked_component_ids':   str(r['ground_truth_component_ids']),
            'linked_component_names': ', '.join(names),
            'llm_reasoning':          str(r.get('llm_reasoning', ''))[:200],
            'VERIFIED_CORRECT':       '',   # <- YES or NO
            'CORRECT_IDS_IF_WRONG':   '',   # <- comma-separated IDs if NO
            'NOTES':                  ''
        })

    new_checklist_df = pd.DataFrame(rows)
    new_checklist_path = f'{OUTPUT_DIR}/verification_checklist_1.xlsx'
    new_checklist_df.to_excel(new_checklist_path, index=False)
    print(f'\nSaved to {new_checklist_path}')
    print('Fill in VERIFIED_CORRECT column then merge with ground_truth_final.json.')
else:
    print('\nAll projects already covered in verification_checklist.xlsx!')

In [ ]:
import json, pandas as pd, ast

# ── Board IDs to exclude (from your dictionary audit) ────────────
BOARD_IDS = {87, 88, 89, 90, 91, 92, 93, 94, 95, 96}

# ── 1. Clean ground_truth_raw.json ───────────────────────────────
with open(f'{OUTPUT_DIR}/ground_truth_raw.json') as f:
    ground_truth = json.load(f)

cleaned = 0
for record in ground_truth:
    before = set(record['ground_truth_component_ids'])
    after  = [i for i in record['ground_truth_component_ids'] if i not in BOARD_IDS]
    if len(before) != len(after):
        record['ground_truth_component_ids'] = after
        record['lib_linked_ids']  = [i for i in record['lib_linked_ids']  if i not in BOARD_IDS]
        record['llm_linked_ids']  = [i for i in record['llm_linked_ids']  if i not in BOARD_IDS]
        cleaned += 1

with open(f'{OUTPUT_DIR}/ground_truth_raw.json', 'w') as f:
    json.dump(ground_truth, f, indent=2)
print(f'ground_truth_raw.json: cleaned board IDs from {cleaned} projects')

# ── Helper to clean ID strings in checklist columns ──────────────
def remove_board_ids_from_str(id_str):
    try:
        ids = ast.literal_eval(str(id_str))
        cleaned = [i for i in ids if int(i) not in BOARD_IDS]
        return str(cleaned)
    except:
        return id_str

# ── 2. Clean verification_checklist.xlsx ─────────────────────────
vc1 = pd.read_excel(f'{OUTPUT_DIR}/verification_checklist.xlsx')
vc1['linked_component_ids'] = vc1['linked_component_ids'].apply(remove_board_ids_from_str)
vc1.to_excel(f'{OUTPUT_DIR}/verification_checklist.xlsx', index=False)
print(f'verification_checklist.xlsx: board IDs removed')

# ── 3. Clean verification_checklist_1.xlsx ───────────────────────
try:
    vc2 = pd.read_excel(f'{OUTPUT_DIR}/verification_checklist_1.xlsx')
    vc2['linked_component_ids'] = vc2['linked_component_ids'].apply(remove_board_ids_from_str)
    vc2.to_excel(f'{OUTPUT_DIR}/verification_checklist_1.xlsx', index=False)
    print(f'verification_checklist_1.xlsx: board IDs removed')
except FileNotFoundError:
    print('verification_checklist_1.xlsx not found — skipping')

# ── 4. Verify ─────────────────────────────────────────────────────
remaining = [
    r for r in ground_truth
    if any(i in BOARD_IDS for i in r['ground_truth_component_ids'])
]
print(f'\nVerification: {len(remaining)} projects still contain board IDs (should be 0)')
print('All three files cleaned and saved.')

In [ ]:
import json, time
from openai import OpenAI

with open(f'{OUTPUT_DIR}/ground_truth_final.json') as f:
    gt = json.load(f)

RATE_LIMIT_IDS = {19, 27, 33, 54, 187, 330, 355, 397, 408}
client = OpenAI()
canonical_list = dictionary.get_canonical_names_list()

for record in gt:
    pid = record['project_id']
    if pid not in RATE_LIMIT_IDS:
        continue

    print(f'Re-running Project {pid}...')
    time.sleep(10)  # wait between calls to avoid hitting rate limit again

    llm_ids, llm_confidence, reasoning = llm_link_components(
        record, canonical_list, record['lib_linked_ids'], client
    )

    all_ids = list(set(record['lib_linked_ids'] + llm_ids))
    record['ground_truth_component_ids'] = all_ids
    record['llm_linked_ids']   = llm_ids
    record['linking_confidence'].update({str(i): llm_confidence.get(i, 'medium') for i in llm_ids})
    record['llm_reasoning']    = reasoning
    print(f'  Done -> IDs: {all_ids}')

with open(f'{OUTPUT_DIR}/ground_truth_final.json', 'w') as f:
    json.dump(gt, f, indent=2)

print('\nAll 9 projects re-linked and saved.')

In [ ]:
with open(f'{OUTPUT_DIR}/pilot_50.json') as f:
    pilot = json.load(f)
pilot_ids = {r['project_id'] for r in pilot}

priority_ids = {19, 27, 33, 54, 125, 328, 281, 122, 187, 355, 61}
overlap = pilot_ids & priority_ids
print(f'Priority projects in your pilot set: {overlap}')

In [ ]:
import pandas as pd
import json

# ── LOAD PROJECT METADATA ─────────────────────────────────────────
metadata_df = pd.read_excel(PROJ_PATH)

# Strip all column names to remove trailing/leading spaces
metadata_df.columns = metadata_df.columns.str.strip()

# Now we can safely create the lookup
desc_map = dict(zip(metadata_df['Project_ID'], metadata_df['Description']))

# JSON paths in OUTPUT_DIR
raw_json_path = OUTPUT_DIR / "ground_truth_raw.json"
final_json_path = OUTPUT_DIR / "ground_truth_final.json"

# Function to update JSON with Description
def update_json_with_description(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    
    missing = 0
    for entry in data:
        pid = entry.get('Project_ID')
        if pid in desc_map:
            entry['Description'] = desc_map[pid]
        else:
            missing += 1
    
    with open(json_path, "w") as f:
        json.dump(data, f, indent=2)
    
    print(f"Updated {json_path.name}. Missing descriptions for {missing} entries.")

# ── UPDATE BOTH JSON FILES ───────────────────────────────────────
update_json_with_description(raw_json_path)
update_json_with_description(final_json_path)

In [ ]:
import pandas as pd
import json
from pathlib import Path

# ── ASSUMING YOUR EXISTING PATHS ───────────────────────────────
# DATA_DIR and OUTPUT_DIR are already defined
# PROJ_PATH points to projects_metadata.xlsx

# ── LOAD AND CLEAN METADATA ────────────────────────────────────
metadata_df = pd.read_excel(PROJ_PATH)

# Strip column names to remove spaces
metadata_df.columns = metadata_df.columns.str.strip()

# Ensure Project_ID is string (to match JSON)
metadata_df['Project_ID'] = metadata_df['Project_ID'].astype(str)

# Create lookup: Project_ID → Description
desc_map = dict(zip(metadata_df['Project_ID'], metadata_df['Description']))

# ── FUNCTION TO UPDATE JSONS ───────────────────────────────────
def update_json_with_description(json_path):
    with open(json_path, "r") as f:
        data = json.load(f)
    
    # Convert JSON Project_IDs to string for consistent matching
    for entry in data:
        entry['Project_ID'] = str(entry.get('Project_ID'))
    
    # Track missing IDs
    missing_ids = []
    updated_count = 0
    
    for entry in data:
        pid = entry.get('Project_ID')
        if pid in desc_map:
            entry['Description'] = desc_map[pid]
            updated_count += 1
        else:
            missing_ids.append(pid)
    
    # Save updated JSON
    with open(json_path, "w") as f:
        json.dump(data, f, indent=2)
    
    print(f"✅ Updated {json_path.name}: {updated_count} entries updated.")
    if missing_ids:
        print(f"⚠️ {len(missing_ids)} Project_IDs missing in Excel:")
        print(missing_ids)

# ── UPDATE BOTH JSON FILES ───────────────────────────────────
raw_json_path = OUTPUT_DIR / "ground_truth_raw.json"
final_json_path = OUTPUT_DIR / "ground_truth_final.json"

update_json_with_description(raw_json_path)
update_json_with_description(final_json_path)

### 0.8 Final Validation — Confirm Pilot Set is Ready

In [ ]:
# Load final ground truth and extract verified pilot set
with open(f'{OUTPUT_DIR}/ground_truth_final.json') as f:
    gt_final = json.load(f)

verified_pilot = [r for r in gt_final if r['manually_verified']]
print(f'Verified pilot projects ready: {len(verified_pilot)}')

# Check all have code files
missing_code = []
for r in verified_pilot:
    code_path = r.get('code_path', '')
    if code_path and not os.path.exists(code_path):
        # Try resolving relative to base dir
        alt_path = os.path.join(CODE_BASE_DIR, os.path.basename(code_path))
        if not os.path.exists(alt_path):
            missing_code.append(r['project_id'])

if missing_code:
    print(f'WARNING: {len(missing_code)} projects have missing code files: {missing_code[:10]}')
else:
    print('All code files accessible.')

# Complexity breakdown of pilot
from collections import Counter
complexity_dist = Counter(r['complexity'] for r in verified_pilot)
print(f'\nPilot complexity distribution: {dict(complexity_dist)}')

# Average components per project
avg = sum(len(r['ground_truth_component_ids']) for r in verified_pilot) / len(verified_pilot) if verified_pilot else 0
print(f'Average ground truth components per project: {avg:.2f}')

print('\n=== STAGE 0 + STAGE 1 COMPLETE ===')
print('You are ready to build and run the pipeline (Stage 2).')

---
## Summary of Outputs

After running all cells, you will have the following files in your `benchmark_outputs/` directory:

| File | Description |
|------|-------------|
| `dictionary_audit_issues.csv` | Any schema issues found in components_dictionary.xlsx |
| `ground_truth_raw.json` | All 435 projects with auto-linked component IDs |
| `zero_component_projects.csv` | Projects where linking found nothing — needs manual review |
| `pilot_50.json` | Stratified 50-project pilot set |
| `verification_checklist.xlsx` | Fill this in manually then run Step 0.7 |
| `ground_truth_final.json` | Final verified ground truth — used in all experiments |

**Next step:** Stage 2 — build the LLM pipeline and run Task 1 (Component Extraction) on the pilot set.